# Notebook 2 – Tấn công đối kháng (FGSM, PGD)

Người phụ trách: **Nguyễn Quang Huy**.

Chạy tấn công white-box, untargeted trên checkpoint baseline ResNet-18 CIFAR-10.
FGSM và PGD dùng cùng tập test cố định từ get_eval_subset(as_tensors=True), seed và
cấu hình trong src/config.py. Ảnh luôn ở [0,1]; Normalize nằm trong mô hình.

Đầu ra trong results/: attack_metrics.json, attack_metrics.csv,
attack_predictions.csv, attack_accuracy.png và ảnh minh họa FGSM/PGD.
Không huấn luyện lại baseline. Giao diện dùng chung: docs/INTERFACE.md.


## 0. Chuẩn bị môi trường

- **Colab:** bật GPU, mount Drive và dùng checkpoint tại MyDrive/AdvML_Project/checkpoints.
  Notebook clone repo nếu chưa có; nếu repo đã tồn tại thì dùng mã hiện tại.
  Repo mới được clone từ REPO_BRANCH trong cell setup. Khi đã gộp nhánh, đổi biến này thành main.
- **Local:** chọn Python của .venv và mở notebook từ thư mục repo hoặc notebooks/.
  Cài thư viện theo requirements.txt nếu môi trường chưa có.
- Dữ liệu và checkpoint local được tìm trong repo trước, sau đó trong thư mục
  AdvML_Project nằm cạnh repo. Thông số thí nghiệm vẫn lấy từ config.py.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/Eastern2k4/adversarial-attack-defense.git"
REPO_BRANCH = "feat/attack-code"  # Đổi thành "main" sau khi nhóm gộp nhánh.
IN_COLAB = "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    Path("/content/drive/MyDrive/AdvML_Project").mkdir(parents=True, exist_ok=True)
    REPO_DIR = Path("/content/adversarial-attack-defense")
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)], check=True)
else:
    cwd = Path.cwd().resolve()
    REPO_DIR = next(
        (root for root in (cwd, *cwd.parents)
         if (root / "src/config.py").is_file() and (root / "notebooks").is_dir()),
        None,
    )
    if REPO_DIR is None:
        raise FileNotFoundError("Hãy mở notebook từ thư mục repo hoặc notebooks/.")

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
print("Repo:", REPO_DIR)


In [ ]:
import csv
import hashlib
import json
import platform
import shutil
import time
from functools import partial

import matplotlib.pyplot as plt
import numpy as np
import torch
import torchvision

from src import config
from src.attacks import fgsm, pgd
from src.data import get_eval_indices, get_eval_subset
from src.models import build_model
from src.utils import accuracy_under_attack, load_checkpoint, set_seed

device = config.get_device()
if device.type == "cpu":
    # Chỉ điều chỉnh số luồng tính toán; không đổi cấu hình thí nghiệm.
    torch.set_num_threads(min(torch.get_num_threads(), 4))
set_seed(config.SEED, deterministic=True)
EVAL_BATCH_SIZE = config.BATCH_SIZE
RESULTS_DIR = config.RESULTS_DIR
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
generated_files = []
print("Device:", device, "| PyTorch:", torch.__version__)


## 1. Nạp baseline và tập đánh giá chung

Chọn đúng checkpoint baseline_resnet18.pth. Nếu dữ liệu CIFAR-10 đã có sẵn thì
không tải lại. Dùng num_workers=0 trên CPU để chạy được cả trên Windows.

Accuracy sạch và accuracy sau tấn công trong notebook đều đo trên cùng
config.N_EVAL ảnh, không trộn accuracy toàn bộ 10.000 ảnh test vào bảng này.


In [ ]:
project_dirs = [config.DRIVE_PROJECT_DIR] if IN_COLAB else [REPO_DIR.parent / "AdvML_Project"]
checkpoint_candidates = [
    config.CHECKPOINT_DIR / config.BASELINE_CKPT,
    *(root / "checkpoints" / config.BASELINE_CKPT for root in project_dirs),
]
CHECKPOINT_PATH = next((path for path in checkpoint_candidates if path.is_file()), None)
if CHECKPOINT_PATH is None:
    raise FileNotFoundError(
        "Không tìm thấy baseline. Các đường dẫn đã kiểm tra:\n"
        + "\n".join(str(path) for path in checkpoint_candidates)
    )

data_candidates = [
    config.DATA_DIR,
    *(root / "cifar-10-python" for root in project_dirs),
    *(root / "data" for root in project_dirs),
    *project_dirs,
]
DATA_DIR = next(
    (path for path in data_candidates if (path / "cifar-10-batches-py/test_batch").is_file()),
    config.DATA_DIR,
)
if not (DATA_DIR / "cifar-10-batches-py/test_batch").is_file():
    archive = next(
        (root / "cifar-10-python.tar.gz" for root in project_dirs
         if (root / "cifar-10-python.tar.gz").is_file()),
        None,
    )
    if archive is not None:
        DATA_DIR.mkdir(parents=True, exist_ok=True)
        destination = DATA_DIR / archive.name
        if archive.resolve() != destination.resolve() and not destination.exists():
            shutil.copy2(archive, destination)

x_eval, y_eval = get_eval_subset(
    as_tensors=True,
    data_dir=DATA_DIR,
    download=not (DATA_DIR / "cifar-10-batches-py/test_batch").is_file(),
    num_workers=config.NUM_WORKERS if device.type == "cuda" else 0,
)
assert x_eval.shape == (config.N_EVAL, 3, 32, 32)
assert y_eval.shape == (config.N_EVAL,)
assert 0 <= x_eval.min().item() <= x_eval.max().item() <= 1

model = build_model().to(device)
load_checkpoint(CHECKPOINT_PATH, model, map_location=device)
model.eval()
# Chỉ đánh giá: vẫn tính được gradient theo ảnh khi trọng số không cần gradient.
model.requires_grad_(False)
print("Checkpoint:", CHECKPOINT_PATH)
print("Dữ liệu:", DATA_DIR)
print(f"Tập đánh giá: {len(y_eval)} ảnh | seed={config.SEED} | batch={EVAL_BATCH_SIZE}")


## 2. Đo accuracy, ASR và chi phí tạo nhiễu

- **Accuracy:** tỷ lệ dự đoán đúng trên toàn bộ tập đánh giá, qua accuracy_under_attack().
- **ASR:** số ảnh sạch dự đoán đúng nhưng sau tấn công bị sai / số ảnh sạch dự đoán đúng.
  Không tính lỗi có sẵn của baseline thành thành công tấn công.
- **Thời gian tạo nhiễu:** chỉ thời gian gọi FGSM/PGD; đồng bộ thiết bị trước/sau khi đo.
  Thời gian đánh giá tổng cộng được ghi riêng và phụ thuộc phần cứng.
- Kiểm tra detach, kích thước, device/dtype, khoảng pixel và L∞ của từng batch;
  dung sai số thực cho L∞ là 1e-6.

Hook tạm thời chỉ thu logits của lượt đánh giá trong no_grad(), bỏ qua các lượt
tính gradient bên trong thuật toán. Nhờ vậy ASR dùng chính dự đoán của hàm
đánh giá chung, không phải chạy mô hình thêm một lần.


In [ ]:
def synchronize():
    if device.type == "cuda":
        torch.cuda.synchronize(device)
    elif device.type == "mps":
        torch.mps.synchronize()


def evaluate_with_predictions(attack_fn=None):
    collected = []

    def collect_logits(module, inputs, logits):
        if not torch.is_grad_enabled():
            collected.append(logits.detach().cpu())

    handle = model.register_forward_hook(collect_logits)
    try:
        accuracy = accuracy_under_attack(
            model, x_eval, y_eval, attack_fn=attack_fn, batch_size=EVAL_BATCH_SIZE
        )
    finally:
        handle.remove()
    logits = torch.cat(collected)
    assert logits.shape == (len(y_eval), config.NUM_CLASSES)
    probabilities = logits.softmax(dim=1)
    predictions = logits.argmax(dim=1)
    expected_accuracy = 100.0 * (predictions == y_eval).sum().item() / len(y_eval)
    assert abs(accuracy - expected_accuracy) < 1e-9
    return accuracy, predictions, probabilities


class AttackMonitor:
    def __init__(self, attack_fn, eps, keep_images=False):
        self.attack_fn = attack_fn
        self.eps = eps
        self.keep_images = keep_images
        self.images = []
        self.generation_time_sec = 0.0
        self.max_linf = 0.0
        self.pixel_min = 1.0
        self.pixel_max = 0.0
        self.count = 0

    def __call__(self, attack_model, xb, yb):
        synchronize()
        started = time.perf_counter()
        x_adv = self.attack_fn(attack_model, xb, yb)
        synchronize()
        self.generation_time_sec += time.perf_counter() - started
        assert x_adv.shape == xb.shape
        assert x_adv.device == xb.device and x_adv.dtype == xb.dtype
        assert not x_adv.requires_grad and x_adv.grad_fn is None
        assert torch.isfinite(x_adv).all().item()

        linf = (x_adv - xb).abs().max().item()
        pixel_min, pixel_max = x_adv.min().item(), x_adv.max().item()
        assert linf <= self.eps + 1e-6, (linf, self.eps)
        assert 0.0 <= pixel_min <= pixel_max <= 1.0
        self.max_linf = max(self.max_linf, linf)
        self.pixel_min = min(self.pixel_min, pixel_min)
        self.pixel_max = max(self.pixel_max, pixel_max)
        if self.keep_images:
            self.images.append(x_adv.detach().cpu())
        self.count += len(yb)
        print(f"  Đã tạo nhiễu: {self.count}/{len(y_eval)} ảnh", flush=True)
        return x_adv


def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def save_metrics():
    destination = RESULTS_DIR / "attack_metrics.json"
    temporary = destination.with_suffix(".json.tmp")
    temporary.write_text(
        json.dumps(metrics, indent=2, ensure_ascii=False, allow_nan=False) + "\n",
        encoding="utf-8",
    )
    temporary.replace(destination)


In [ ]:
print("Đang đánh giá ảnh sạch...", flush=True)
synchronize()
started = time.perf_counter()
clean_acc, clean_predictions, clean_probabilities = evaluate_with_predictions()
synchronize()
clean_eval_time_sec = time.perf_counter() - started
clean_correct = clean_predictions.eq(y_eval)
n_clean_correct = int(clean_correct.sum().item())
print(f"Clean accuracy: {clean_acc:.2f}% ({n_clean_correct}/{len(y_eval)})", flush=True)


## 3. Chạy FGSM và PGD ở mọi epsilon

FGSM tăng cross-entropy theo dấu gradient trong một bước. PGD dùng
config.PGD_STEPS bước, alpha=config.PGD_ALPHA, random_start=True và **một lần
khởi tạo**. Cả hai dùng nhãn thật, targeted=False.

Đặt lại seed config.SEED trước mỗi cấu hình để chạy lại độc lập. Thông số này
được lưu trong JSON; kết quả PGD có thể lệch nhẹ nếu phần phòng thủ dùng một
chuỗi RNG hoặc batch size khác. Không mặc định PGD luôn mạnh hơn FGSM:
nhận xét dựa trên số liệu đo được.

JSON được lưu sau từng cấu hình với completed=False; chỉ khi đủ mọi phép đo
mới chuyển completed=True. Nếu phiên chạy bị ngắt, chạy lại cell này để đo lại.


In [ ]:
eval_indices = get_eval_indices()
eps_keys = [f"{round(eps * 255)}/255" for eps in config.EPSILONS]
illustration_eps = max(config.EPSILONS)
metrics = {
    "model": CHECKPOINT_PATH.stem,
    "clean_acc": clean_acc,
    "fgsm": {},
    "pgd": {},
    "asr": {"fgsm": {}, "pgd": {}},
    "details": {"fgsm": {}, "pgd": {}},
    "evaluation": {
        "completed": False,
        "dataset": "CIFAR-10 test",
        "n_eval": len(y_eval),
        "seed": config.SEED,
        "batch_size": EVAL_BATCH_SIZE,
        "clean_correct": n_clean_correct,
        "clean_eval_time_sec": clean_eval_time_sec,
        "eval_indices_sha256": hashlib.sha256(
            json.dumps(eval_indices, separators=(",", ":")).encode("utf-8")
        ).hexdigest(),
        "accuracy_unit": "percent",
        "asr_definition": "clean_correct_and_adversarial_wrong / clean_correct * 100",
        "asr_undefined_value": None,
    },
    "attack_config": {
        "norm": "Linf",
        "epsilons": eps_keys,
        "targeted": False,
        "pgd_alpha": config.PGD_ALPHA,
        "pgd_steps": config.PGD_STEPS,
        "random_start": True,
        "restarts": 1,
        "seed_per_attack": config.SEED,
        "linf_tolerance": 1e-6,
    },
    "provenance": {
        "checkpoint": CHECKPOINT_PATH.name,
        "checkpoint_sha256": file_sha256(CHECKPOINT_PATH),
        "attacks_py_sha256": file_sha256(REPO_DIR / "src/attacks.py"),
        "device": str(device),
        "python": platform.python_version(),
        "torch": torch.__version__,
        "torchvision": torchvision.__version__,
        "numpy": np.__version__,
        "cpu_threads": torch.get_num_threads(),
        "deterministic": True,
    },
}
prediction_columns = {
    "test_index": eval_indices,
    "true_label": y_eval.tolist(),
    "clean_pred": clean_predictions.tolist(),
}
illustrations = {}
save_metrics()

for eps, eps_key in zip(config.EPSILONS, eps_keys):
    attacks = {
        "fgsm": partial(fgsm, eps=eps, targeted=False),
        "pgd": partial(
            pgd, eps=eps, alpha=config.PGD_ALPHA,
            steps=config.PGD_STEPS, random_start=True, targeted=False,
        ),
    }
    for attack_name, attack_fn in attacks.items():
        print(f"\n{attack_name.upper()} | eps={eps_key}", flush=True)
        set_seed(config.SEED, deterministic=True)
        monitor = AttackMonitor(attack_fn, eps, keep_images=(eps == illustration_eps))
        synchronize()
        started = time.perf_counter()
        accuracy, predictions, probabilities = evaluate_with_predictions(monitor)
        synchronize()
        total_time_sec = time.perf_counter() - started
        assert monitor.count == len(y_eval)
        successful = int((clean_correct & predictions.ne(y_eval)).sum().item())
        asr = 100.0 * successful / n_clean_correct if n_clean_correct else None

        metrics[attack_name][eps_key] = accuracy
        metrics["asr"][attack_name][eps_key] = asr
        metrics["details"][attack_name][eps_key] = {
            "correct": int(predictions.eq(y_eval).sum().item()),
            "successful_attacks_on_clean_correct": successful,
            "generation_time_sec": monitor.generation_time_sec,
            "generation_time_ms_per_image": 1000.0 * monitor.generation_time_sec / len(y_eval),
            "total_eval_time_sec": total_time_sec,
            "max_linf": monitor.max_linf,
            "pixel_min": monitor.pixel_min,
            "pixel_max": monitor.pixel_max,
        }
        prediction_columns[f"{attack_name}_{eps_key}"] = predictions.tolist()
        if eps == illustration_eps:
            illustrations[attack_name] = {
                "images": torch.cat(monitor.images),
                "predictions": predictions,
                "probabilities": probabilities,
            }
        save_metrics()
        asr_text = f"{asr:.2f}%" if asr is not None else "không xác định"
        print(
            f"{attack_name.upper()} eps={eps_key}: accuracy={accuracy:.2f}% | "
            f"ASR={asr_text} | tạo nhiễu={monitor.generation_time_sec:.2f}s | "
            f"Linf={monitor.max_linf:.8f}",
            flush=True,
        )

metrics["evaluation"]["completed"] = True
save_metrics()
generated_files.append(RESULTS_DIR / "attack_metrics.json")
assert all(parameter.grad is None for parameter in model.parameters())
print("\nĐã hoàn tất toàn bộ cấu hình tấn công.", flush=True)


## 4. Bảng số liệu và dự đoán từng ảnh

attack_metrics.csv chứa accuracy, ASR, thời gian và giới hạn nhiễu cho từng cấu hình.
attack_predictions.csv lưu chỉ số test gốc, nhãn thật, dự đoán sạch và dự đoán sau
từng tấn công để kiểm tra lại các chỉ số. Các dự đoán là nhãn CIFAR-10 (0–9).


In [ ]:
rows = [{
    "attack": "clean", "epsilon": "0", "accuracy_pct": clean_acc,
    "asr_pct": None, "correct": n_clean_correct,
    "successful_attacks_on_clean_correct": 0,
    "generation_time_sec": 0.0, "generation_time_ms_per_image": 0.0,
    "total_eval_time_sec": clean_eval_time_sec, "max_linf": 0.0,
    "pixel_min": x_eval.min().item(), "pixel_max": x_eval.max().item(),
}]
for eps_key in eps_keys:
    for attack_name in ("fgsm", "pgd"):
        rows.append({
            "attack": attack_name,
            "epsilon": eps_key,
            "accuracy_pct": metrics[attack_name][eps_key],
            "asr_pct": metrics["asr"][attack_name][eps_key],
            **metrics["details"][attack_name][eps_key],
        })

table_path = RESULTS_DIR / "attack_metrics.csv"
with table_path.open("w", newline="", encoding="utf-8") as stream:
    writer = csv.DictWriter(stream, fieldnames=list(rows[0]))
    writer.writeheader()
    writer.writerows(rows)

prediction_path = RESULTS_DIR / "attack_predictions.csv"
with prediction_path.open("w", newline="", encoding="utf-8") as stream:
    writer = csv.writer(stream)
    writer.writerow(prediction_columns)
    writer.writerows(zip(*prediction_columns.values()))

generated_files.extend([table_path, prediction_path])
print(f"{'Attack':<8} {'Epsilon':<8} {'Accuracy (%)':>13} {'ASR (%)':>10} {'Gen (s)':>10}")
for row in rows:
    asr_text = f"{row['asr_pct']:.2f}" if row["asr_pct"] is not None else "-"
    print(
        f"{row['attack']:<8} {row['epsilon']:<8} {row['accuracy_pct']:>13.2f} "
        f"{asr_text:>10} {row['generation_time_sec']:>10.2f}"
    )


## 5. Accuracy theo epsilon

Hai đường bắt đầu tại cùng accuracy ảnh sạch (epsilon=0). Trục hoành biểu diễn
epsilon theo đơn vị 1/255. PNG được lưu ở độ phân giải 200 dpi để đưa vào báo cáo.


In [ ]:
epsilon_pixels = [0] + [round(eps * 255) for eps in config.EPSILONS]
fig, ax = plt.subplots(figsize=(8, 5))
for attack_name, marker in (("fgsm", "o"), ("pgd", "s")):
    values = [clean_acc] + [metrics[attack_name][key] for key in eps_keys]
    label = "FGSM" if attack_name == "fgsm" else f"PGD-{config.PGD_STEPS}"
    ax.plot(epsilon_pixels, values, marker=marker, linewidth=2, label=label)
    for epsilon, accuracy in zip(epsilon_pixels[1:], values[1:]):
        ax.annotate(
            f"{accuracy:.1f}", (epsilon, accuracy),
            xytext=(5, 7 if attack_name == "fgsm" or accuracy < 5 else -14),
            textcoords="offset points", fontsize=8,
        )
ax.set(
    xlabel="Epsilon (1/255)", ylabel="Accuracy (%)",
    title=f"Baseline ResNet-18 | CIFAR-10 | n={len(y_eval)}, seed={config.SEED}",
    xticks=epsilon_pixels, ylim=(0, 100),
)
ax.grid(alpha=0.3)
ax.legend()
fig.tight_layout()
chart_path = RESULTS_DIR / "attack_accuracy.png"
fig.savefig(chart_path, dpi=200, bbox_inches="tight")
generated_files.append(chart_path)
plt.show()
plt.close(fig)


## 6. Ảnh gốc / nhiễu / ảnh đối kháng

Với mỗi thuật toán, chọn tối đa 5 ảnh baseline đoán đúng nhưng bị tấn công làm sai
ở epsilon lớn nhất trong config. Nếu không có ca thành công thì ghi rõ và minh họa
ảnh có sẵn, không coi đó là tấn công thành công.

Ảnh gốc và ảnh đối kháng giữ đúng pixel [0,1]. **Chỉ cột nhiễu được phóng đại**:
màu hiển thị bằng 0.5 + delta/(2*epsilon), với xám 0.5 biểu diễn nhiễu bằng 0.
Nhãn dự đoán và xác suất softmax được ghi trên từng ảnh.


In [ ]:
for attack_name, result in illustrations.items():
    predictions = result["predictions"]
    successful_indices = torch.where(clean_correct & predictions.ne(y_eval))[0].tolist()
    selected = successful_indices[:5]
    selection_note = "successful attacks (clean correct -> adversarial wrong)"
    if not selected:
        selected = torch.where(clean_correct)[0].tolist()[:5] or list(range(min(5, len(y_eval))))
        selection_note = "no successful attack found; illustrative samples"

    fig, axes = plt.subplots(len(selected), 3, figsize=(10, 2.6 * len(selected)), squeeze=False)
    for row, index in enumerate(selected):
        original = x_eval[index]
        adversarial = result["images"][index]
        delta = adversarial - original
        noise_display = (0.5 + delta / (2 * illustration_eps)).clamp(0, 1)
        true_label = config.CLASSES[int(y_eval[index])]
        clean_label = config.CLASSES[int(clean_predictions[index])]
        adv_label = config.CLASSES[int(predictions[index])]
        clean_confidence = clean_probabilities[index, clean_predictions[index]].item()
        adv_confidence = result["probabilities"][index, predictions[index]].item()
        titles = [
            f"Test #{eval_indices[index]} | true: {true_label}\n"
            f"Clean: {clean_label} ({clean_confidence:.1%})",
            f"Scaled noise\nLinf={delta.abs().max().item():.5f}",
            f"{attack_name.upper()}: {adv_label} ({adv_confidence:.1%})\n"
            f"epsilon={round(illustration_eps * 255)}/255",
        ]
        for column, (image, title) in enumerate(zip((original, noise_display, adversarial), titles)):
            axes[row, column].imshow(image.permute(1, 2, 0).numpy(), interpolation="nearest")
            axes[row, column].set_title(title, fontsize=9)
            axes[row, column].axis("off")
    fig.suptitle(f"{attack_name.upper()}: {selection_note}", fontsize=11)
    fig.tight_layout(rect=(0, 0, 1, 0.97))
    example_path = RESULTS_DIR / f"attack_examples_{attack_name}.png"
    fig.savefig(example_path, dpi=200, bbox_inches="tight")
    generated_files.append(example_path)
    plt.show()
    plt.close(fig)


## 7. Nhận xét dựa trên số liệu

Cell dưới in các số đo chính để đưa vào phần thảo luận. Khi viết báo cáo, giải thích
sự giảm accuracy theo ngân sách nhiễu, khác biệt giữa một bước FGSM và PGD lặp,
và chi phí tính toán tương ứng. ASR chỉ tính trên các ảnh sạch được phân loại đúng.

Các kết quả mô tả đúng cấu hình PGD đang chạy (số bước, alpha, một khởi tạo), không
phải phép chứng minh độ bền vững của mô hình. Khi ghép với kết quả phòng thủ,
dùng cùng tập đánh giá và phân biệt rõ clean accuracy 1.000 ảnh với 10.000 ảnh.


In [ ]:
largest_key = f"{round(illustration_eps * 255)}/255"
print(f"Baseline đúng {n_clean_correct}/{len(y_eval)} ảnh sạch: {clean_acc:.2f}%.")
for attack_name in ("fgsm", "pgd"):
    attacked_acc = metrics[attack_name][largest_key]
    detail = metrics["details"][attack_name][largest_key]
    print(
        f"{attack_name.upper()} tại eps={largest_key}: accuracy {attacked_acc:.2f}%, "
        f"giảm {clean_acc - attacked_acc:.2f} điểm phần trăm; "
        f"làm sai {detail['successful_attacks_on_clean_correct']}/{n_clean_correct} "
        "ảnh vốn được dự đoán đúng."
    )
    print(f"  Thời gian tạo nhiễu: {detail['generation_time_sec']:.2f}s trên {device}.")
gap = metrics["fgsm"][largest_key] - metrics["pgd"][largest_key]
print(f"Accuracy FGSM trừ accuracy PGD-{config.PGD_STEPS}: {gap:.2f} điểm phần trăm.")
print("JSON hoàn chỉnh:", metrics["evaluation"]["completed"])


## 8. Lưu kết quả

Trên Colab, chỉ các file do notebook này tạo ra được sao lưu sang
MyDrive/AdvML_Project/results để giữ lại khi runtime kết thúc.
Local giữ kết quả trong results/ của repo. Không commit dữ liệu hay trọng số.
Trước khi commit notebook, dùng Clear All Outputs.


In [ ]:
if IN_COLAB:
    backup_dir = config.DRIVE_PROJECT_DIR / "results"
    backup_dir.mkdir(parents=True, exist_ok=True)
    for path in generated_files:
        shutil.copy2(path, backup_dir / path.name)
    print("Đã sao lưu kết quả lên:", backup_dir)
for path in generated_files:
    print(path.relative_to(REPO_DIR))
